# Практическая работа #5: Редактирование SQL таблиц (Сквозная модель)
## Методический ориентир и эталонное решение (для преподавателя)

Блокнот содержит сквозное решение задач всех трех уровней сложности по единой модели: **все темы (DDL, ограничения, внешние ключи, импорт CSV через append, UPDATE, DELETE, транзакции) представлены на каждом уровне**, но с разной степенью разветвленности условий и глубины логики:

* **🟢 Easy:** 2 связанные таблицы (`members` -> `game_sessions`). Создание схемы DDL, импорт CSV через `append`, проверка `FOREIGN KEY`, `CHECK`, `UNIQUE`, точечные `UPDATE`, безопасный `DELETE`, двухшаговая транзакция с фиксированной суммой (`COMMIT` и `ROLLBACK`).
* **🟡 Medium:** 3 связанные таблицы (`hardware_pcs`, `members`, `game_sessions`). Схема DDL с двумя внешними ключами, импорт CSV в правильном порядке, массовые `UPDATE` с выражением и датами, удаление по условию, блокировка удаления родителей, динамическая транзакция (`hours * 150`).
* **🔴 Hard:** 3 связанные таблицы с `ON DELETE CASCADE`. Сложная транзакция со связыванием тарифов ПК, демонстрация автоматической каскадной очистки зависимых строк, аналитический `UPDATE` со вложенным запросом.
* **Критерии оценивания** на «5», «4», «3», «2».


In [24]:
import sqlite3
import pandas as pd
import numpy as np
import pathlib

# Определение рабочей папки с наборами вариантов
base_dir = pathlib.Path(".")
if not (base_dir / "easy").exists():
    base_dir = pathlib.Path(r"d:\GitHub\TexhProg-34\05 - Редактирование SQL таблиц")

print("Рабочая директория данных:", base_dir.resolve())
print("Версия Pandas:", pd.__version__)
print("Версия SQLite3:", sqlite3.sqlite_version)


Рабочая директория данных: D:\GitHub\TexhProg-34\05 - Редактирование SQL таблиц
Версия Pandas: 3.0.2
Версия SQLite3: 3.50.4


---
# 🟢 Часть 1. Эталонное решение Уровня Easy

### Задание 1.1 и 1.2: Создание схемы DDL и импорт данных из CSV (вариант 1)


In [ ]:
# Инициализация чистой базы данных
db_easy_path = base_dir / "club_easy.db"
try:
    if db_easy_path.exists():
        db_easy_path.unlink()
except PermissionError:
    pass

conn_easy = sqlite3.connect(db_easy_path)
conn_easy.execute("PRAGMA foreign_keys = ON;")
cur_easy = conn_easy.cursor()

# 1. Создание структуры таблиц DDL
cur_easy.executescript("""
DROP TABLE IF EXISTS game_sessions;
DROP TABLE IF EXISTS members;

CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL UNIQUE,
    email         TEXT UNIQUE,
    city          TEXT NOT NULL DEFAULT 'Москва',
    balance       REAL NOT NULL DEFAULT 0 CHECK (balance >= 0),
    registered_at TEXT NOT NULL DEFAULT (CURRENT_DATE)
);

CREATE TABLE game_sessions (
    session_id   INTEGER PRIMARY KEY,
    member_id    INTEGER NOT NULL,
    game_title   TEXT NOT NULL,
    start_time   TEXT NOT NULL,
    hours_played REAL NOT NULL CHECK (hours_played > 0),
    status       TEXT NOT NULL DEFAULT 'active' CHECK (status IN ('active', 'completed')),
    FOREIGN KEY (member_id) REFERENCES members(member_id)
);
""")

# 2. Импорт CSV в строгой последовательности (родители -> дочерняя)
var1_easy = base_dir / "easy" / "variant_01"
pd.read_csv(var1_easy / "members.csv", sep=";").to_sql("members", conn_easy, if_exists="append", index=False)
pd.read_csv(var1_easy / "game_sessions.csv", sep=";").to_sql("game_sessions", conn_easy, if_exists="append", index=False)
conn_easy.commit()

print("✅ Схема DDL создана, данные варианта 1 успешно загружены через append:")
display(pd.read_sql_query("""
    SELECT 'members' AS [Таблица], COUNT(*) AS [Записей] FROM members
    UNION ALL
    SELECT 'game_sessions', COUNT(*) FROM game_sessions;
""", conn_easy))


✅ Схема DDL создана, данные варианта 1 успешно загружены через append:


,Таблица,Записей
0,members,100
1,game_sessions,200


### Задание 1.3. Тестирование ограничения CHECK (отрицательное время игры)


In [26]:
try:
    cur_easy.execute("""
        INSERT INTO game_sessions (member_id, game_title, start_time, hours_played)
        VALUES (?, ?, ?, ?);
    """, (1, 'Counter-Strike 2', '2026-03-15 14:00:00', -1.0))
    conn_easy.commit()
except sqlite3.IntegrityError as e:
    print(f"✅ Перехвачено ожидаемое нарушение ограничения CHECK: {e}")


✅ Перехвачено ожидаемое нарушение ограничения CHECK: CHECK constraint failed: hours_played > 0


### Задание 1.4. Тестирование FOREIGN KEY (сессия на несуществующего клиента)


In [27]:
try:
    cur_easy.execute("""
        INSERT INTO game_sessions (member_id, game_title, start_time, hours_played)
        VALUES (?, ?, ?, ?);
    """, (99999, 'Dota 2', '2026-03-15 15:00:00', 2.0))
    conn_easy.commit()
except sqlite3.IntegrityError as e:
    print(f"✅ Перехвачена ожидаемая ошибка отсутствующего внешнего ключа: {e}")


✅ Перехвачена ожидаемая ошибка отсутствующего внешнего ключа: FOREIGN KEY constraint failed


### Задание 1.5. Тестирование ограничения UNIQUE (дубликат никнейма)


In [28]:
existing_nick = cur_easy.execute("SELECT nickname FROM members LIMIT 1").fetchone()[0]

try:
    cur_easy.execute("""
        INSERT INTO members (nickname, email, balance)
        VALUES (?, ?, ?);
    """, (existing_nick, 'duplicate@mail.ru', 100))
    conn_easy.commit()
except sqlite3.IntegrityError as e:
    print(f"✅ Перехвачено ожидаемое нарушение уникальности (UNIQUE): {e}")


✅ Перехвачено ожидаемое нарушение уникальности (UNIQUE): UNIQUE constraint failed: members.nickname


### Задание 1.6. UPDATE: пополнение депозита клиента (+300 руб.)


In [29]:
target_id = 1
print(f"Баланс клиента {target_id} ДО пополнения:")
display(pd.read_sql_query("SELECT member_id, nickname, balance FROM members WHERE member_id = ?;", conn_easy, params=(target_id,)))

cur_easy.execute("""
    UPDATE members
    SET balance = balance + ?
    WHERE member_id = ?;
""", (300.0, target_id))
conn_easy.commit()
print(f"Обновлено строк: {cur_easy.rowcount}")

print(f"Баланс клиента {target_id} ПОСЛЕ пополнения:")
display(pd.read_sql_query("SELECT member_id, nickname, balance FROM members WHERE member_id = ?;", conn_easy, params=(target_id,)))


Баланс клиента 1 ДО пополнения:


,member_id,nickname,balance
0,1,CyberKnight_319,2500.0


Обновлено строк: 1
Баланс клиента 1 ПОСЛЕ пополнения:


,member_id,nickname,balance
0,1,CyberKnight_319,2800.0


### Задание 1.7. UPDATE: завершение сессии


In [30]:
sess_to_complete = 1005
cur_easy.execute("UPDATE game_sessions SET status = 'completed' WHERE session_id = ?;", (sess_to_complete,))
conn_easy.commit()
print(f"Сессия {sess_to_complete} переведена в completed (строк обновлено: {cur_easy.rowcount}).")
display(pd.read_sql_query("SELECT session_id, member_id, status FROM game_sessions WHERE session_id = ?;", conn_easy, params=(sess_to_complete,)))


Сессия 1005 переведена в completed (строк обновлено: 1).


,session_id,member_id,status
0,1005,78,completed


### Задание 1.8. DELETE: удаление одной сессии и проверка защиты внешнего ключа


In [31]:
# 1. Безопасное удаление дочерней сессии
cnt_before = cur_easy.execute("SELECT COUNT(*) FROM game_sessions;").fetchone()[0]
cur_easy.execute("DELETE FROM game_sessions WHERE session_id = 1001;")
conn_easy.commit()
cnt_after = cur_easy.execute("SELECT COUNT(*) FROM game_sessions;").fetchone()[0]
print(f"Сессий ДО: {cnt_before}, ПОСЛЕ: {cnt_after} (разница: {cnt_before - cnt_after})")

# 2. Попытка удалить родителя, за которым числятся сессии
parent_id = cur_easy.execute("SELECT member_id FROM game_sessions LIMIT 1;").fetchone()[0]
try:
    cur_easy.execute("DELETE FROM members WHERE member_id = ?;", (parent_id,))
    conn_easy.commit()
except sqlite3.IntegrityError as e:
    print(f"✅ База отклонила удаление родительского клиента с активными сессиями: {e}")


Сессий ДО: 200, ПОСЛЕ: 199 (разница: 1)
✅ База отклонила удаление родительского клиента с активными сессиями: FOREIGN KEY constraint failed


### Задание 1.9. Управление транзакциями: успешная фиксация (COMMIT)


In [32]:
# 1. Находим клиента с достаточным балансом (>= 200) и его активную сессию
client_rich = cur_easy.execute("""
    SELECT m.member_id, m.balance, s.session_id 
    FROM members m
    JOIN game_sessions s ON m.member_id = s.member_id
    WHERE m.balance >= 200 AND s.status = 'active'
    LIMIT 1;
""").fetchone()

if client_rich:
    m_id, old_bal, s_id = client_rich
    print(f"ДО транзакции: Клиент ID={m_id}, Баланс={old_bal} руб., Сессия ID={s_id} (active)")

    # Атомарный блок транзакции
    try:
        cur_easy.execute("UPDATE members SET balance = balance - 200 WHERE member_id = ?;", (m_id,))
        cur_easy.execute("UPDATE game_sessions SET status = 'completed' WHERE session_id = ?;", (s_id,))
        conn_easy.commit()
        print("✅ Транзакция успешно зафиксирована (COMMIT)!")
    except Exception as e:
        conn_easy.rollback()
        print(f"Сбой: {e}")

    new_bal = cur_easy.execute("SELECT balance FROM members WHERE member_id = ?;", (m_id,)).fetchone()[0]
    new_status = cur_easy.execute("SELECT status FROM game_sessions WHERE session_id = ?;", (s_id,)).fetchone()[0]
    print(f"ПОСЛЕ транзакции: Баланс={new_bal} руб., Статус сессии={new_status}")


ДО транзакции: Клиент ID=8, Баланс=1000.0 руб., Сессия ID=1007 (active)
✅ Транзакция успешно зафиксирована (COMMIT)!
ПОСЛЕ транзакции: Баланс=800.0 руб., Статус сессии=completed


### Задание 1.10. Управление транзакциями: аварийный откат (ROLLBACK)


In [33]:
# 2. Находим клиента с нулевым балансом (< 200)
client_poor = cur_easy.execute("""
    SELECT member_id, balance 
    FROM members 
    WHERE balance < 200 
    LIMIT 1;
""").fetchone()

if client_poor:
    p_id, p_bal = client_poor
    p_sess = cur_easy.execute("SELECT session_id FROM game_sessions WHERE member_id = ? LIMIT 1;", (p_id,)).fetchone()[0]
    print(f"Клиент {p_id} (баланс {p_bal} руб.) пытается списать 200 руб.")
    
    try:
        cur_easy.execute("UPDATE game_sessions SET status = 'completed' WHERE session_id = ?;", (p_sess,))
        cur_easy.execute("UPDATE members SET balance = balance - 200 WHERE member_id = ?;", (p_id,))
        conn_easy.commit()
    except Exception as e:
        conn_easy.rollback()
        print(f"✅ Перехвачена ошибка: {e}")
        print("🔄 Выполнен ROLLBACK. Все изменения отменены!")
        
    bal_after_rollback = cur_easy.execute("SELECT balance FROM members WHERE member_id = ?;", (p_id,)).fetchone()[0]
    sess_status = cur_easy.execute("SELECT status FROM game_sessions WHERE session_id = ?;", (p_sess,)).fetchone()[0]
    print(f"Проверка: баланс остался {bal_after_rollback} руб., статус сессии остался {sess_status}.")

conn_easy.close()


Клиент 4 (баланс 150.0 руб.) пытается списать 200 руб.
✅ Перехвачена ошибка: CHECK constraint failed: balance >= 0
🔄 Выполнен ROLLBACK. Все изменения отменены!
Проверка: баланс остался 150.0 руб., статус сессии остался completed.


---
# 🟡 Часть 2. Эталонное решение Уровня Medium

### Задание 2.1 и 2.2: Создание схемы DDL с двумя внешними ключами и импорт CSV (вариант 1)


In [34]:
db_med_path = base_dir / "club_medium.db"
try:
    if db_med_path.exists():
        db_med_path.unlink()
except PermissionError:
    pass

conn_med = sqlite3.connect(db_med_path)
conn_med.execute("PRAGMA foreign_keys = ON;")
cur_med = conn_med.cursor()

cur_med.executescript("""
DROP TABLE IF EXISTS game_sessions;
DROP TABLE IF EXISTS members;
DROP TABLE IF EXISTS hardware_pcs;

CREATE TABLE hardware_pcs (
    pc_id       INTEGER PRIMARY KEY,
    pc_number   TEXT NOT NULL UNIQUE,
    zone_name   TEXT NOT NULL CHECK (zone_name IN ('Standard', 'Bootcamp', 'VIP', 'PS5')),
    hourly_rate REAL NOT NULL CHECK (hourly_rate > 0),
    gpu         TEXT NOT NULL,
    status      TEXT NOT NULL DEFAULT 'Active' CHECK (status IN ('Active', 'Maintenance', 'Reserved'))
);

CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL UNIQUE,
    email         TEXT UNIQUE,
    city          TEXT NOT NULL DEFAULT 'Москва',
    balance       REAL NOT NULL DEFAULT 0 CHECK (balance >= 0),
    registered_at TEXT NOT NULL DEFAULT (CURRENT_DATE)
);

CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER NOT NULL,
    pc_id          INTEGER NOT NULL,
    game_title     TEXT NOT NULL,
    start_time     TEXT NOT NULL,
    duration_hours REAL NOT NULL CHECK (duration_hours > 0 AND duration_hours <= 24),
    status         TEXT NOT NULL DEFAULT 'active' CHECK (status IN ('active', 'completed', 'cancelled')),
    client_rating  INTEGER CHECK (client_rating BETWEEN 1 AND 5),
    FOREIGN KEY (member_id) REFERENCES members(member_id),
    FOREIGN KEY (pc_id) REFERENCES hardware_pcs(pc_id)
);
""")

var1_med = base_dir / "medium" / "variant_01"
pd.read_csv(var1_med / "hardware_pcs.csv", sep=";").to_sql("hardware_pcs", conn_med, if_exists="append", index=False)
pd.read_csv(var1_med / "members.csv", sep=";").to_sql("members", conn_med, if_exists="append", index=False)
pd.read_csv(var1_med / "game_sessions.csv", sep=";").to_sql("game_sessions", conn_med, if_exists="append", index=False)
conn_med.commit()

print("✅ Схема DDL создана, данные Medium успешно импортированы через append:")
display(pd.read_sql_query("""
    SELECT 'hardware_pcs' AS [Таблица], COUNT(*) AS [Записей] FROM hardware_pcs
    UNION ALL
    SELECT 'members', COUNT(*) FROM members
    UNION ALL
    SELECT 'game_sessions', COUNT(*) FROM game_sessions;
""", conn_med))


✅ Схема DDL создана, данные Medium успешно импортированы через append:


,Таблица,Записей
0,hardware_pcs,30
1,members,100
2,game_sessions,300


### Задание 2.3. Проверка внешнего ключа оборудования (pc_id = 99999)


In [35]:
try:
    cur_med.execute("""
        INSERT INTO game_sessions (member_id, pc_id, game_title, start_time, duration_hours, status)
        VALUES (?, ?, ?, ?, ?, ?);
    """, (1, 99999, 'Dota 2', '2026-03-15 14:00:00', 3.0, 'completed'))
    conn_med.commit()
except sqlite3.IntegrityError as e:
    print(f"✅ Перехвачена ожидаемая ошибка отсутствующего внешнего ключа ПК: {e}")


✅ Перехвачена ожидаемая ошибка отсутствующего внешнего ключа ПК: FOREIGN KEY constraint failed


### Задание 2.4. Проверка ограничения CHECK (рейтинг client_rating = 10)


In [36]:
try:
    cur_med.execute("""
        INSERT INTO game_sessions (member_id, pc_id, game_title, start_time, duration_hours, status, client_rating)
        VALUES (?, ?, ?, ?, ?, ?, ?);
    """, (1, 1, 'Valorant', '2026-03-15 14:00:00', 1.0, 'completed', 10))
    conn_med.commit()
except sqlite3.IntegrityError as e:
    print(f"✅ Перехвачено нарушение ограничения рейтинга (CHECK BETWEEN 1 AND 5): {e}")


✅ Перехвачено нарушение ограничения рейтинга (CHECK BETWEEN 1 AND 5): CHECK constraint failed: client_rating BETWEEN 1 AND 5


### Задание 2.5. Массовый UPDATE: закрытие устаревших сессий и индексация VIP-тарифов


In [37]:
# 1. Завершение сессий до 2026-03-01
cutoff_date = '2026-03-01'
cur_med.execute("""
    UPDATE game_sessions
    SET status = 'completed'
    WHERE status = 'active' AND start_time < ?;
""", (cutoff_date,))
conn_med.commit()
print(f"Успешно переведено в completed: {cur_med.rowcount} сессий.")

# 2. Индексация тарифов зоны VIP на 10%
cur_med.execute("""
    UPDATE hardware_pcs
    SET hourly_rate = ROUND(hourly_rate * 1.10)
    WHERE zone_name = 'VIP';
""")
conn_med.commit()
print(f"Обновлено тарифов зоны VIP: {cur_med.rowcount}")
display(pd.read_sql_query("SELECT pc_id, pc_number, zone_name, hourly_rate FROM hardware_pcs WHERE zone_name = 'VIP';", conn_med))


Успешно переведено в completed: 21 сессий.
Обновлено тарифов зоны VIP: 5


,pc_id,pc_number,zone_name,hourly_rate
0,23,PC-23,VIP,418.0
1,24,PC-24,VIP,440.0
2,25,PC-25,VIP,440.0
3,26,PC-26,VIP,440.0
4,27,PC-27,VIP,462.0


### Задание 2.6. DELETE: удаление отмененных сессий и проверка ссылочной защиты


In [38]:
# 1. Удаление cancelled сессий
cur_med.execute("DELETE FROM game_sessions WHERE status = 'cancelled';")
conn_med.commit()
print(f"Удалено отмененных сессий: {cur_med.rowcount}")

# 2. Попытка удалить ПК, за которым закреплены сессии
parent_pc = cur_med.execute("SELECT pc_id FROM game_sessions LIMIT 1;").fetchone()[0]
try:
    cur_med.execute("DELETE FROM hardware_pcs WHERE pc_id = ?;", (parent_pc,))
    conn_med.commit()
except sqlite3.IntegrityError as e:
    print(f"✅ Удаление компьютера заблокировано защитой внешнего ключа: {e}")


Удалено отмененных сессий: 24
✅ Удаление компьютера заблокировано защитой внешнего ключа: FOREIGN KEY constraint failed


### Задание 2.7. Транзакция с динамическим расчетом стоимости (duration_hours * 150)


In [39]:
target_sess = cur_med.execute("""
    SELECT s.session_id, s.member_id, s.duration_hours, m.balance
    FROM game_sessions s
    JOIN members m ON s.member_id = m.member_id
    WHERE s.status = 'active' AND m.balance >= (s.duration_hours * 150)
    LIMIT 1;
""").fetchone()

if target_sess:
    s_id, m_id, dur, bal = target_sess
    total_cost = dur * 150.0
    print(f"Сессия {s_id}: длительность {dur}ч, сумма к списанию {total_cost} руб. Баланс клиента {bal} руб.")

    try:
        cur_med.execute("UPDATE members SET balance = balance - ? WHERE member_id = ?;", (total_cost, m_id))
        cur_med.execute("UPDATE game_sessions SET status = 'completed' WHERE session_id = ?;", (s_id,))
        conn_med.commit()
        print("✅ Динамическая транзакция успешно зафиксирована (COMMIT)!")
    except Exception as e:
        conn_med.rollback()
        print(f"Ошибка транзакции: {e}")

    new_bal = cur_med.execute("SELECT balance FROM members WHERE member_id = ?;", (m_id,)).fetchone()[0]
    print(f"Новый баланс клиента {m_id}: {new_bal} руб.")

# Демонстрация отката при недостатке средств
poor_sess = cur_med.execute("""
    SELECT s.session_id, s.member_id, s.duration_hours, m.balance
    FROM game_sessions s
    JOIN members m ON s.member_id = m.member_id
    WHERE s.status = 'active' AND m.balance < (s.duration_hours * 150)
    LIMIT 1;
""").fetchone()

if poor_sess:
    ps_id, pm_id, pdur, pbal = poor_sess
    req_cost = pdur * 150.0
    print(f"Попытка списать {req_cost} руб. у клиента с балансом {pbal} руб.")
    try:
        cur_med.execute("UPDATE game_sessions SET status = 'completed' WHERE session_id = ?;", (ps_id,))
        cur_med.execute("UPDATE members SET balance = balance - ? WHERE member_id = ?;", (req_cost, pm_id))
        conn_med.commit()
    except Exception as e:
        conn_med.rollback()
        print(f"🔄 Сработал откат транзакции (ROLLBACK): {e}")

conn_med.close()


Сессия 1014: длительность 1.0ч, сумма к списанию 150.0 руб. Баланс клиента 1000.0 руб.
✅ Динамическая транзакция успешно зафиксирована (COMMIT)!
Новый баланс клиента 71: 850.0 руб.
Попытка списать 150.0 руб. у клиента с балансом 0.0 руб.
🔄 Сработал откат транзакции (ROLLBACK): CHECK constraint failed: balance >= 0


---
# 🔴 Часть 3. Эталонное решение Уровня Hard

### Задание 3.1 и 3.2: Схема DDL со связью ON DELETE CASCADE и импорт CSV (вариант 1)


In [40]:
db_hard_path = base_dir / "club_hard.db"
try:
    if db_hard_path.exists():
        db_hard_path.unlink()
except PermissionError:
    pass

conn_hard = sqlite3.connect(db_hard_path)
conn_hard.execute("PRAGMA foreign_keys = ON;")
cur_hard = conn_hard.cursor()

cur_hard.executescript("""
DROP TABLE IF EXISTS game_sessions;
DROP TABLE IF EXISTS members;
DROP TABLE IF EXISTS hardware_pcs;

CREATE TABLE hardware_pcs (
    pc_id       INTEGER PRIMARY KEY,
    pc_number   TEXT NOT NULL UNIQUE,
    zone_name   TEXT NOT NULL CHECK (zone_name IN ('Standard', 'Bootcamp', 'VIP', 'PS5')),
    hourly_rate REAL NOT NULL CHECK (hourly_rate > 0),
    gpu         TEXT NOT NULL,
    status      TEXT NOT NULL DEFAULT 'Active' CHECK (status IN ('Active', 'Maintenance', 'Reserved'))
);

CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL UNIQUE,
    email         TEXT UNIQUE,
    city          TEXT NOT NULL DEFAULT 'Москва',
    balance       REAL NOT NULL DEFAULT 0 CHECK (balance >= 0),
    registered_at TEXT NOT NULL DEFAULT (CURRENT_DATE)
);

CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER,
    pc_id          INTEGER NOT NULL,
    game_title     TEXT NOT NULL,
    start_time     TEXT NOT NULL,
    duration_hours REAL NOT NULL CHECK (duration_hours > 0 AND duration_hours <= 24),
    status         TEXT NOT NULL DEFAULT 'active' CHECK (status IN ('active', 'completed', 'cancelled')),
    client_rating  INTEGER CHECK (client_rating BETWEEN 1 AND 5),
    FOREIGN KEY (member_id) REFERENCES members(member_id) ON DELETE CASCADE,
    FOREIGN KEY (pc_id) REFERENCES hardware_pcs(pc_id)
);
""")

var1_hard = base_dir / "hard" / "variant_01"
pd.read_csv(var1_hard / "hardware_pcs.csv", sep=";").to_sql("hardware_pcs", conn_hard, if_exists="append", index=False)
pd.read_csv(var1_hard / "members.csv", sep=";").to_sql("members", conn_hard, if_exists="append", index=False)
pd.read_csv(var1_hard / "game_sessions.csv", sep=";").to_sql("game_sessions", conn_hard, if_exists="append", index=False)
conn_hard.commit()

print("✅ Схема DDL создана, данные Hard успешно импортированы:")
display(pd.read_sql_query("""
    SELECT 'hardware_pcs' AS [Таблица], COUNT(*) AS [Записей] FROM hardware_pcs
    UNION ALL
    SELECT 'members', COUNT(*) FROM members
    UNION ALL
    SELECT 'game_sessions', COUNT(*) FROM game_sessions;
""", conn_hard))


✅ Схема DDL создана, данные Hard успешно импортированы:


,Таблица,Записей
0,hardware_pcs,30
1,members,100
2,game_sessions,300


### Задание 3.3. Комплексная транзакция: списание стоимости по тарифу ПК (COMMIT / ROLLBACK)


In [41]:
# Найдем активную сессию с достаточным балансом с учетом реального тарифа ПК
session_row = cur_hard.execute("""
    SELECT s.session_id, s.member_id, s.duration_hours, h.hourly_rate, m.balance
    FROM game_sessions s
    JOIN hardware_pcs h ON s.pc_id = h.pc_id
    JOIN members m ON s.member_id = m.member_id
    WHERE s.status = 'active' AND m.balance >= (s.duration_hours * h.hourly_rate)
    LIMIT 1;
""").fetchone()

if session_row:
    sess_id, mem_id, dur, rate, old_bal = session_row
    cost = dur * rate
    print(f"Сессия {sess_id}: клиент ID={mem_id}, длительность={dur}ч, тариф={rate} руб./ч. Стоимость к списанию: {cost} руб.")
    print(f"Баланс ДО транзакции: {old_bal} руб.")

    try:
        cur_hard.execute("UPDATE game_sessions SET status = 'completed' WHERE session_id = ?;", (sess_id,))
        cur_hard.execute("UPDATE members SET balance = balance - ? WHERE member_id = ?;", (cost, mem_id))
        conn_hard.commit()
        print("✅ Транзакция успешно зафиксирована (COMMIT)!")
    except Exception as e:
        conn_hard.rollback()
        print(f"Ошибка транзакции: {e}")

    new_bal = cur_hard.execute("SELECT balance FROM members WHERE member_id = ?;", (mem_id,)).fetchone()[0]
    sess_status = cur_hard.execute("SELECT status FROM game_sessions WHERE session_id = ?;", (sess_id,)).fetchone()[0]
    print(f"ПОСЛЕ транзакции: Баланс клиента={new_bal} руб., Статус сессии={sess_status}")


Сессия 1002: клиент ID=17, длительность=3.0ч, тариф=150.0 руб./ч. Стоимость к списанию: 450.0 руб.
Баланс ДО транзакции: 1500.0 руб.
✅ Транзакция успешно зафиксирована (COMMIT)!
ПОСЛЕ транзакции: Баланс клиента=1050.0 руб., Статус сессии=completed


### Задание 3.4. Аварийный откат транзакции при нехватке средств (ROLLBACK)


In [42]:
poor_session = cur_hard.execute("""
    SELECT s.session_id, s.member_id, s.duration_hours, h.hourly_rate, m.balance
    FROM game_sessions s
    JOIN hardware_pcs h ON s.pc_id = h.pc_id
    JOIN members m ON s.member_id = m.member_id
    WHERE s.status = 'active' AND m.balance < (s.duration_hours * h.hourly_rate)
    LIMIT 1;
""").fetchone()

if poor_session:
    p_sess_id, p_mem_id, p_dur, p_rate, p_old_bal = poor_session
    p_cost = p_dur * p_rate
    print(f"Клиент ID={p_mem_id} (баланс {p_old_bal} руб.) пытается оплатить сессию стоимостью {p_cost} руб.")

    try:
        cur_hard.execute("UPDATE game_sessions SET status = 'completed' WHERE session_id = ?;", (p_sess_id,))
        if p_old_bal < p_cost:
            raise ValueError(f"Недостаточно средств на балансе! Требуется {p_cost}, доступно {p_old_bal}.")
        cur_hard.execute("UPDATE members SET balance = balance - ? WHERE member_id = ?;", (p_cost, p_mem_id))
        conn_hard.commit()
    except Exception as e:
        conn_hard.rollback()
        print(f"✅ Перехвачено исключение: {e}")
        print("🔄 Выполнен ROLLBACK. Все действия отменены!")

    check_bal = cur_hard.execute("SELECT balance FROM members WHERE member_id = ?;", (p_mem_id,)).fetchone()[0]
    check_status = cur_hard.execute("SELECT status FROM game_sessions WHERE session_id = ?;", (p_sess_id,)).fetchone()[0]
    print(f"Проверка после отката: баланс клиента={check_bal} руб. (не изменился), статус сессии={check_status} (осталась active)")


Клиент ID=2 (баланс 0.0 руб.) пытается оплатить сессию стоимостью 810.0 руб.
✅ Перехвачено исключение: Недостаточно средств на балансе! Требуется 810.0, доступно 0.0.
🔄 Выполнен ROLLBACK. Все действия отменены!
Проверка после отката: баланс клиента=0.0 руб. (не изменился), статус сессии=active (осталась active)


### Задание 3.5. Демонстрация каскадного удаления (ON DELETE CASCADE)


In [43]:
target_client = cur_hard.execute("""
    SELECT member_id, COUNT(*) 
    FROM game_sessions 
    GROUP BY member_id 
    HAVING COUNT(*) >= 2 
    LIMIT 1;
""").fetchone()

if target_client:
    c_id, sess_cnt = target_client
    print(f"Клиент ID={c_id} имеет {sess_cnt} сессий в таблице game_sessions.")

    # Удаляем самого клиента из members:
    cur_hard.execute("DELETE FROM members WHERE member_id = ?;", (c_id,))
    conn_hard.commit()
    print(f"Клиент ID={c_id} удален из members.")

    # Проверим, что стало с его сессиями:
    rem_sess_cnt = cur_hard.execute("SELECT COUNT(*) FROM game_sessions WHERE member_id = ?;", (c_id,)).fetchone()[0]
    print(f"Сессий клиента ID={c_id} после удаления родителя: {rem_sess_cnt} (каскадно удалены СУБД!)")


Клиент ID=1 имеет 3 сессий в таблице game_sessions.
Клиент ID=1 удален из members.
Сессий клиента ID=1 после удаления родителя: 0 (каскадно удалены СУБД!)


### Задание 3.6. Аналитический UPDATE со вложенным запросом


In [44]:
q_bonus = """
UPDATE members
SET balance = balance + 200
WHERE city IN ('Москва', 'Санкт-Петербург')
  AND member_id IN (
      SELECT DISTINCT member_id 
      FROM game_sessions 
      WHERE status = 'completed' AND member_id IS NOT NULL
  );
"""
cur_hard.execute(q_bonus)
conn_hard.commit()
print(f"Начислен бонус клиентам двух столиц: обновлено строк = {cur_hard.rowcount}")

display(pd.read_sql_query("""
    SELECT member_id, nickname, city, balance 
    FROM members 
    WHERE city IN ('Москва', 'Санкт-Петербург') 
    LIMIT 5;
""", conn_hard))

conn_hard.close()


Начислен бонус клиентам двух столиц: обновлено строк = 47


,member_id,nickname,city,balance
0,2,Vortex_117,Москва,200.0
1,3,Turbo_366,Москва,700.0
2,4,Matrix_580,Москва,4200.0
3,8,Storm_455,Санкт-Петербург,4200.0
4,10,Titan_209,Санкт-Петербург,4200.0


---
# 📊 Критерии оценивания практической работы

| Оценка | Требования к выполнению |
|:---:|---|
| **«5» (Отлично)** | Выбран уровень Medium или Hard. Создана корректная DDL-схема со всеми типами и ограничениями (`PRIMARY KEY`, `NOT NULL`, `CHECK`, `UNIQUE`, `DEFAULT`, `FOREIGN KEY`). Данные импортированы через `append` с соблюдением порядка. Все тесты целостности оформлены через `try/except`. Все `UPDATE` и `DELETE` содержат `WHERE` и параметризацию `?`. Реализована транзакция с проверкой `COMMIT` и `ROLLBACK`. Оформлены краткие выводы. |
| **«4» (Хорошо)** | Выполнен уровень Easy в полном объеме, либо уровень Medium/Hard с 1–2 некритичными недочетами (например, забыт `PRAGMA foreign_keys = ON;`, часть условий передана f-строками вместо `?`, либо отсутствует обработка ошибки при транзакции). |
| **«3» (Удовлетворительно)** | Выполнен уровень Easy, но есть грубые нарушения: данные загружены через `if_exists='replace'` (утеряна схема DDL), отсутствуют проверки `IntegrityError`, или транзакция не содержит отката при ошибке. |
| **«2» (Неудовлетворительно)** | Схема не спроектирована, SQL-запросы вызывают фатальные синтаксические ошибки, нарушены базовые принципы целостности базы данных. |
